# Module 3: Deploy It (Managed Deep Agents)

> Part 2 of the workshop, ~10 min. Prerequisite: Modules 1–2 (the agent is registered as `survey-feedback-agent`, and `survey_agent/` builds).

1. **Deploy** the `survey_agent/` project with `mda deploy`, bound to the agent's **development** environment.
2. **Call it through the API**. The run pauses before posting, and the paused thread shows up for approval in **Agent Inbox**.
3. **Promote it to staging**: the same project deployed again, bound to the **staging** environment.

The deployments and their traces all live under one agent: the Overview lists its deployments, and Tracing splits traffic by environment.

## Setup

In [ ]:
import os, sys, time
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)

# Agent addressing and LANGSMITH_PROJECT can't be combined: with both set, the SDK
# can't pick a destination and drops the trace.
if os.environ.pop("LANGSMITH_PROJECT", None):
    print("Note: ignoring LANGSMITH_PROJECT in this kernel (conflicts with agent addressing).")

import warnings
# Keep outputs readable: agent addressing is in beta (one-time notice), and
# list_runs() is deprecated but supported until 2027.
warnings.filterwarnings("ignore", message="Addressing runs with `langsmith.address` is in beta")
warnings.filterwarnings("ignore", message=r"list_runs\(\) is deprecated")

import langsmith as ls
from langsmith import Client, uuid7

from utils.models import model
from utils.agent_registry import ensure_agent, environment_project_name

AGENT_ID = "survey-feedback-agent"   # the agent's identifier in LangSmith (lowercase, hyphens)
ENVIRONMENT = "development"
dev_address = ls.agent_address(AGENT_ID, ENVIRONMENT)

client = Client()
# Models go through the LangSmith LLM Gateway. `langsmith:<provider>/<model>` models
# read LANGSMITH_GATEWAY_API_KEY (then LANGSMITH_API_KEY); reuse the workshop's gateway key.
if os.environ.get("LANGSMITH_API_KEY_GATEWAY"):
    os.environ.setdefault("LANGSMITH_GATEWAY_API_KEY", os.environ["LANGSMITH_API_KEY_GATEWAY"])
for key in ["LANGSMITH_API_KEY", "LANGSMITH_API_KEY_GATEWAY"]:
    print(f"{key:26s} set: {bool(os.environ.get(key))}")
import subprocess

survey_project = project_root / "survey_agent"
ensure_agent(client, AGENT_ID, name="Survey Feedback Agent")
# survey_agent/ is its own uv project; drop the notebook's VIRTUAL_ENV so uv uses that project's env.
mda_env = {k: v for k, v in os.environ.items() if k != "VIRTUAL_ENV"}
print(subprocess.run(["uv", "run", "mda", "--version"], cwd=survey_project, env=mda_env,
                     capture_output=True, text=True).stdout.strip())

## 3.1 What `mda deploy` ships

`mda deploy` compiles `survey_agent/` and sends each part to the right place:

| Project input | Goes to |
|---|---|
| `instructions.md`, `skills/**` | Context Hub (editable without a rebuild) |
| `agent.py`, `subagents.py`, `tools/`, `data/`, `identity.py`, `sandbox/` | The deployment's build |
| `.env` | Deploy auth, plus hosted deployment secrets for non-reserved values (e.g. `SURVEY_WEBHOOK_URL`). The model runs through the LLM Gateway (`langsmith:openai/gpt-5.6-luna`), so no provider key ships. Never archived into the build |
| `schedules/**` (none yet) | LangSmith cron jobs, once the deployment is live |

The deployment runs on Agent Server, which supplies the checkpointer. The approval gate needs no extra setup.

## 3.2 Give the project its keys

`mda` reads `survey_agent/.env` first, then the shell. The next cell writes only the keys the deployment needs into that file, which is gitignored. You can also edit it by hand.

In [ ]:
keys = ["LANGSMITH_API_KEY", "LANGSMITH_WORKSPACE_ID", "LANGSMITH_ENDPOINT", "LANGSMITH_GATEWAY_API_KEY", "SURVEY_WEBHOOK_URL"]
lines = [f'{k}="{os.environ[k]}"' for k in keys if os.environ.get(k)]
(survey_project / ".env").write_text("\n".join(lines) + "\n")
print("Wrote survey_agent/.env with:", [line.split("=")[0] for line in lines])

## 3.3 Deploy to development

`--agent-environment development` binds the deployment to the agent's Development environment, and the deployment name is used as the agent ID, so `--name survey-feedback-agent` targets the agent from Module 1. Traces from the deployment land there automatically.

> `mda deploy` can stop to ask questions, such as a Context Hub conflict on a later deploy. Running it in a **terminal** from `survey_agent/` lets you answer them. The cell below works for a first deploy; for non-interactive re-deploys, add `--context-strategy overwrite` or `--context-strategy keep-hub`.

In [ ]:
!cd "{survey_project}" && env -u VIRTUAL_ENV uv run mda deploy --name {AGENT_ID} --agent-environment development

When the deploy finishes, the CLI prints two URLs: **Agent Server**, the deployment's API URL, and **Dashboard**. Paste the Agent Server URL below. You can also find it under the agent → **Overview** → **Deployments**. To follow runtime logs from a terminal, run `uv run mda logs` in `survey_agent/`.

In [ ]:
DEV_URL = "https://<your-dev-deployment>.us.langgraph.app"  # paste the "Agent Server" URL printed by mda deploy

## 3.4 Call it through the API

The deployment requires a LangSmith API key from your workspace (`identity.py`). The `langgraph_sdk` client sends it as `x-api-key`. The graph ID is the definition's `name`, `survey-feedback-agent`. The run pauses before `post_themes_to_channel`, and the thread's status becomes `interrupted`.

In [ ]:
from langgraph_sdk import get_client

dev = get_client(url=DEV_URL, api_key=os.environ["LANGSMITH_API_KEY"])

thread = await dev.threads.create(metadata={"workshop": "survey-feedback"})
thread_id = thread["thread_id"]
await dev.runs.wait(thread_id, AGENT_ID, input={"messages": [{
    "role": "user",
    "content": "Analyze the latest survey export and post the summary to #survey-insights.",
}]})

thread = await dev.threads.get(thread_id)
print("status:", thread["status"])
for interrupts in (thread.get("interrupts") or {}).values():
    for intr in interrupts:
        for action in intr["value"]["action_requests"]:
            print(f"\nWaiting for approval: {action['name']} -> {action['args']['channel']}\n")
            print(action["args"]["message"])

## 3.5 Approve in Agent Inbox

The paused thread is waiting for a human. In LangSmith, open **Agent Inbox** for the development deployment, select the thread, and review the `post_themes_to_channel` action. From there you can **Approve**, **Edit** the message, or **Reject** it with feedback for the agent.

The same decision can come from any client through the API. Run the next cell if you'd rather approve from code. (Skip it if you already decided in Agent Inbox.)

In [ ]:
await dev.runs.wait(thread_id, AGENT_ID, command={"resume": {"decisions": [{"type": "approve"}]}})
thread = await dev.threads.get(thread_id)
print("status:", thread["status"])
print(thread["values"]["messages"][-1]["content"])

Open the agent → **Tracing** → **Development**. The deployed runs are there, including the paused run and the resumed run on the same thread.

**Also on every Managed Deep Agents deployment:** `<DEV_URL>/mcp` serves the agent as an MCP tool, so Claude Code or another agent can call it with the same `x-api-key` header.

## 3.6 Promote to staging

Promotion is the same project deployed again with `--agent-environment staging`. It becomes a second deployment under the same agent, reporting into its own environment, so staging traffic gets its own dashboards, evaluators and alerts. In a real rollout you'd promote after Module 4's experiment passes.

In [ ]:
!cd "{survey_project}" && env -u VIRTUAL_ENV uv run mda deploy --name {AGENT_ID} --agent-environment staging

In [ ]:
STAGING_URL = "https://<your-staging-deployment>.us.langgraph.app"  # paste the staging API URL

staging = get_client(url=STAGING_URL, api_key=os.environ["LANGSMITH_API_KEY"])
t = await staging.threads.create()
out = await staging.runs.wait(t["thread_id"], AGENT_ID, input={"messages": [{
    "role": "user", "content": "What is NPS by channel? Answer in two sentences.",
}]})
print(out["messages"][-1]["content"])

Open the agent's **Overview**: both deployments are listed under **Deployments**, and **Tracing** shows each one's traffic in its own environment.

## Recap

- One CLI command per environment: `uv run mda deploy --name survey-feedback-agent --agent-environment development|staging`.
- Instructions and skills went to Context Hub; code, data, and declarations went into the build; `.env` values became deployment secrets.
- The API run paused before posting, and the approval happened in Agent Inbox (or through the API).
- Development and staging deployments and their traces all sit under one agent.

**Next, Modules 4–6:** evaluate and review, monitor and alert, and bring your own agents (ADK included).